# 01 — Tokenization Deep Dive: How Text Becomes Numbers

> **Orbit -1 (Foundations)** · **Domain**: Foundations · **Difficulty**: Beginner · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/castalia/blob/main/foundations/01_tokenization_deep_dive.ipynb)

**Prerequisites**:
- [00_how_llms_work.ipynb](00_how_llms_work.ipynb) — Understanding the LLM pipeline

**What you will learn**:
- Why tokenization matters: it shapes cost, context limits, and model capability
- How BPE (Byte-Pair Encoding) works — implemented from scratch
- How different tokenizers handle the same text differently
- The impact of tokenization on multilingual text, code, numbers, and edge cases
- How to count tokens for cost estimation and context budgeting

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Why tokenization is the first thing you must understand**
- Understand **BPE from scratch**
- Understand **Modern tokenizer variants**
- Understand **Tokenization and cost: counting what you'll pay for**
- Understand **Tokenization gotchas: what can go wrong**


In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" "accelerate>=1.6.0" "bitsandbytes>=0.45.0" \
    "torch>=2.0" "tiktoken>=0.9.0" "sentencepiece>=0.2.0" "matplotlib>=3.8"

# Mount Google Drive for model caching
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, re, collections, json
from typing import Optional
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Cache Hugging Face models to Drive so we don't re-download each session
CACHE_DIR = "/content/drive/MyDrive/models"
os.environ["HF_HOME"] = CACHE_DIR
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Load tokenizers ───────────────────────────────────────────────
from transformers import AutoTokenizer
import tiktoken

print("Loading Qwen3-8B tokenizer …")
qwen_tok = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-8B", cache_dir=CACHE_DIR, trust_remote_code=True
)

print("Loading Llama-3.1-8B tokenizer …")
llama_tok = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.1-8B", cache_dir=CACHE_DIR, trust_remote_code=True
)

print("Loading GPT-4 tiktoken (cl100k_base) …")
gpt4_enc = tiktoken.get_encoding("cl100k_base")

print("✅ All tokenizers ready.")

## 1 — Why tokenization is the first thing you must understand

Every interaction with a large language model starts and ends with **tokens**. When you send a prompt to GPT-4, Claude, or Qwen, the first thing that happens is not "understanding" — it is *tokenization*: your text is sliced into a sequence of integer IDs that the model can process as input vectors.

This matters for three concrete reasons:

| Concern | Why tokenization matters |
|---------|-------------------------|
| **Cost** | API providers charge per token. A prompt that is 1,200 tokens at one tokenizer might be 900 at another — a 25% cost difference for identical text. |
| **Context limits** | Models have fixed context windows measured in tokens (e.g., 128K for GPT-4o). Your *text* length doesn't determine whether it fits — your *token* count does. |
| **Capability** | The way a tokenizer splits numbers, code, and non-English text directly affects what the model can "see" as a unit. If "12345" is split into ["123", "45"], the model never sees the full number as one object. |

Let's make this concrete. We'll tokenize the same English sentence with three different tokenizers and observe how they disagree.

In [ ]:
# @title 1.1 — Same sentence, three tokenizers

sentence = "Tokenization determines what the model sees."

def show_tokens(name: str, token_ids: list[int], decode_fn) -> None:
    '''Print token IDs and their decoded surface forms.'''
    pieces = [decode_fn(tid) for tid in token_ids]
    print(f"{name:>12s}  │  {len(token_ids):>3d} tokens  │  {pieces}")

# Qwen3
qwen_ids = qwen_tok.encode(sentence)
show_tokens("Qwen3-8B", qwen_ids, lambda t: qwen_tok.decode([t]))

# GPT-4 (tiktoken)
gpt4_ids = gpt4_enc.encode(sentence)
show_tokens("GPT-4", gpt4_ids, lambda t: gpt4_enc.decode([t]))

# Llama-3.1
llama_ids = llama_tok.encode(sentence, add_special_tokens=False)
show_tokens("Llama-3.1", llama_ids, lambda t: llama_tok.decode([t]))

Notice that even on a simple English sentence the three tokenizers produce **different token counts and boundaries**. The model behind each tokenizer was trained on a different corpus with a different vocabulary, so it learned different merge rules. This means:

- Switching models can change whether your prompt fits in context.
- The same text costs different amounts under different providers.
- Some tokenizers are more efficient for specific languages or domains (code, math).

Let's quantify this across languages.

In [ ]:
# @title 1.2 — Token counts across languages and domains

samples = {
    "English":   "The quick brown fox jumps over the lazy dog.",
    "Chinese":   "快速的棕色狐狸跳过了懒狗。",
    "Arabic":    "الثعلب البني السريع يقفز فوق الكلب الكسول.",
    "Python":    "def fibonacci(n: int) -> int:\n    return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "JSON":      '{"name": "Alice", "scores": [98, 87, 95], "active": true}',
}

print(f"{'Sample':>10s}  │ {'Qwen3':>6s} │ {'GPT-4':>6s} │ {'Llama':>6s} │ {'Chars':>6s}")
print("─" * 58)

for label, text in samples.items():
    q = len(qwen_tok.encode(text))
    g = len(gpt4_enc.encode(text))
    l = len(llama_tok.encode(text, add_special_tokens=False))
    print(f"{label:>10s}  │ {q:>6d} │ {g:>6d} │ {l:>6d} │ {len(text):>6d}")

**Key insight — tokenization is not neutral.** Chinese text typically requires 2–4× more tokens than English text of equivalent meaning, because most tokenizer vocabularies were built primarily from English corpora. This has direct consequences:

- **Cost**: Serving Chinese users can be significantly more expensive per semantic unit.
- **Context**: A Chinese document fills the context window much faster.
- **Capability**: The model processes each Chinese character through more "steps," potentially degrading performance.

These are not abstract concerns — they shape product decisions, pricing models, and language support strategies.

## 2 — BPE from scratch

To understand modern tokenizers, we need to understand the design space they occupy. There are two naïve extremes:

**Character-level encoding** — the vocabulary is tiny (a few hundred symbols), but sequences become extremely long. The word "tokenization" becomes 12 separate tokens. The model must learn to compose characters into meaning, wasting capacity on spelling.

**Word-level encoding** — sequences are short, but the vocabulary is enormous (hundreds of thousands of entries) and *open*: any new word, typo, or technical term creates an out-of-vocabulary (OOV) problem. You can't embed what you can't index.

**Byte-Pair Encoding (BPE)** is the elegant middle ground. The algorithm:

1. Start with a vocabulary of individual bytes (256 entries).
2. Scan the training corpus and find the most frequent *adjacent pair* of tokens.
3. Merge that pair into a single new token. Add it to the vocabulary.
4. Repeat steps 2–3 until the vocabulary reaches a target size (e.g., 32K, 100K, 150K tokens).

The result: common words and subwords get their own tokens, while rare words are decomposed into smaller known pieces. Let's implement this from scratch.

In [ ]:
# @title 2.1 — BPE merge algorithm from scratch

def get_pair_counts(token_sequences: list[list[str]]) -> dict[tuple[str, str], int]:
    '''Count the frequency of every adjacent token pair across all sequences.'''
    counts: dict[tuple[str, str], int] = collections.Counter()
    for seq in token_sequences:
        for i in range(len(seq) - 1):
            counts[(seq[i], seq[i + 1])] += 1
    return counts


def merge_pair(
    token_sequences: list[list[str]], pair: tuple[str, str]
) -> list[list[str]]:
    '''Replace every occurrence of `pair` with its concatenation.'''
    merged_token = pair[0] + pair[1]
    new_sequences = []
    for seq in token_sequences:
        new_seq = []
        i = 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
                new_seq.append(merged_token)
                i += 2
            else:
                new_seq.append(seq[i])
                i += 1
        new_sequences.append(new_seq)
    return new_sequences


def run_bpe(corpus: list[str], num_merges: int) -> tuple[list[list[str]], list[tuple[str, str]]]:
    '''Run BPE on a corpus of strings for `num_merges` iterations.

    Returns the final token sequences and the ordered list of merges.
    '''
    # Initialise: split every word into characters + end-of-word marker
    token_sequences = [list(word) + ["</w>"] for word in corpus]
    merges = []

    for step in range(num_merges):
        pair_counts = get_pair_counts(token_sequences)
        if not pair_counts:
            break
        best_pair = max(pair_counts, key=pair_counts.get)
        token_sequences = merge_pair(token_sequences, best_pair)
        merges.append(best_pair)
        merged_token = best_pair[0] + best_pair[1]
        print(
            f"  Merge {step + 1:>2d}: {str(best_pair):>20s}  →  '{merged_token}'"
            f"   (freq={pair_counts[best_pair]})"
        )

    return token_sequences, merges

# ── Toy corpus ────────────────────────────────────────────────────
corpus = [
    "low", "low", "low", "low", "low",
    "lower", "lower",
    "newest", "newest", "newest", "newest", "newest", "newest",
    "widest", "widest", "widest",
]

print("Corpus:", corpus)
print(f"Unique words: {set(corpus)}\n")
print("Running BPE (10 merges):\n")

final_seqs, merge_rules = run_bpe(corpus, num_merges=10)

In [ ]:
# @title 2.2 — Inspect the learned vocabulary and encode a new word

# Collect every unique token that survived after BPE
vocab = sorted({tok for seq in final_seqs for tok in seq})
print(f"Vocabulary ({len(vocab)} tokens):")
print(vocab)

# ── Encode a new word using the learned merge rules ───────────────
def encode_word(word: str, merges: list[tuple[str, str]]) -> list[str]:
    '''Tokenize a single word using the ordered BPE merge rules.'''
    tokens = list(word) + ["</w>"]
    for pair in merges:
        i = 0
        while i < len(tokens) - 1:
            if (tokens[i], tokens[i + 1]) == pair:
                tokens[i] = tokens[i] + tokens[i + 1]
                del tokens[i + 1]
            else:
                i += 1
    return tokens

test_words = ["lowest", "newest", "higher"]
for w in test_words:
    encoded = encode_word(w, merge_rules)
    print(f"  '{w}' → {encoded}")

**What just happened?**

1. We started with character-level tokens: `['l', 'o', 'w', '</w>']` for "low".
2. The algorithm found that `('e', 's')` was the most frequent pair across our corpus and merged it into `'es'`.
3. It continued merging — `('es', 't')` → `'est'`, `('l', 'o')` → `'lo'`, `('lo', 'w')` → `'low'`, and so on.
4. After 10 merges, common subwords like `'low'`, `'est'`, and `'new'` became single tokens.

When we encoded the *unseen* word "lowest", the algorithm applied merges in order and decomposed it into known pieces: `['low', 'est', '</w>']`. No out-of-vocabulary problem — the word is built from learned subwords.

This is exactly what GPT, Llama, and Qwen tokenizers do, but at scale: they run BPE on billions of text bytes and build vocabularies of 32K–150K tokens.

## 3 — Modern tokenizer variants

Real-world tokenizers build on BPE but diverge in important ways:

| Tokenizer | Used by | Algorithm | Vocabulary size | Key feature |
|-----------|---------|-----------|-----------------|-------------|
| **tiktoken** (cl100k_base) | GPT-4, GPT-4o | BPE on UTF-8 bytes | ~100,257 | Fast Rust implementation; regex-based pre-splitting |
| **SentencePiece** (Unigram) | Llama, Gemma | Unigram language model | ~32,000–128,000 | Treats input as raw byte stream; no pre-tokenization |
| **Qwen tokenizer** | Qwen, Qwen2.5, Qwen3 | BPE with byte fallback | ~151,643 | Very large vocab; strong CJK and code coverage |

The **Unigram** algorithm (used by SentencePiece) works differently from BPE. Instead of building up from characters, it starts with a *large* initial vocabulary and *prunes* tokens that contribute least to the corpus likelihood. The result is similar — subword tokens — but the training dynamics differ.

Let's compare how these tokenizers handle tricky inputs where their design choices surface.

In [ ]:
# @title 3.1 — Tricky inputs: numbers, code, multilingual, emojis, whitespace

tricky_inputs = {
    "Integer":        "12345",
    "Formatted num":  "12,345",
    "Scientific":     "1.23e5",
    "Python func":    "def hello_world():",
    "CamelCase":      "getElementById",
    "Japanese":       "こんにちは世界",
    "Arabic":         "مرحبا",
    "Emojis":         "🎉🔥",
    "Leading space":  " Hello",
    "Tabs/newlines":  "\t\n  indent",
}

def tok_pieces(name, ids, decode_fn):
    '''Return list of decoded token strings.'''
    return [decode_fn(tid) for tid in ids]

print(f"{'Input':>16s}  │ {'Qwen3':>5s} │ {'GPT-4':>5s} │ {'Llama':>5s}")
print("─" * 52)
detail_rows = []

for label, text in tricky_inputs.items():
    q_ids = qwen_tok.encode(text)
    g_ids = gpt4_enc.encode(text)
    l_ids = llama_tok.encode(text, add_special_tokens=False)
    print(f"{label:>16s}  │ {len(q_ids):>5d} │ {len(g_ids):>5d} │ {len(l_ids):>5d}")
    detail_rows.append((label, text, q_ids, g_ids, l_ids))

print("\n── Detailed token pieces (first 5 examples) ─────────────")
for label, text, q_ids, g_ids, l_ids in detail_rows[:5]:
    print(f"\n  {label}: \"{text}\"")
    print(f"    Qwen3 : {tok_pieces('q', q_ids, lambda t: qwen_tok.decode([t]))}")
    print(f"    GPT-4 : {tok_pieces('g', g_ids, lambda t: gpt4_enc.decode([t]))}")
    print(f"    Llama : {tok_pieces('l', l_ids, lambda t: llama_tok.decode([t]))}")

### Observations

- **Numbers**: `"12345"` might be one token or several. When it is split — say into `["123", "45"]` — the model never sees the full number as one unit. This is a root cause of why LLMs struggle with arithmetic: the operands are shattered.
- **Code**: Underscores in `hello_world` are handled differently by each tokenizer. Some keep `_world` together; others split at `_`. CamelCase identifiers like `getElementById` are split at subword boundaries learned during training.
- **Multilingual**: Japanese characters may each be their own token (expensive!) or grouped into subwords depending on vocabulary coverage. Arabic right-to-left text adds another layer of complexity.
- **Emojis**: Multi-byte emoji sequences often decompose into several tokens — sometimes 3–4 tokens per emoji.
- **Whitespace**: A leading space in `" Hello"` is typically a distinct token — `" Hello"` ≠ `"Hello"`. Tabs and newlines also consume tokens.

## 4 — Tokenization and cost: counting what you'll pay for

API pricing is denominated in tokens. As of mid-2025, representative pricing:

| Provider | Model | Input (per 1M tokens) | Output (per 1M tokens) |
|----------|-------|----------------------:|------------------------:|
| OpenAI | GPT-4o | $2.50 | $10.00 |
| OpenAI | GPT-4o-mini | $0.15 | $0.60 |
| Anthropic | Claude Sonnet 4 | $3.00 | $15.00 |
| Alibaba | Qwen3-235B-A22B | $0.70 | $2.80 |

*Prices change frequently — always check official docs.* The key point: **every wasted token costs real money at scale**, and different tokenizers produce different counts for the same text.

Let's build a cost estimator.

In [ ]:
# @title 4.1 — Token cost calculator

def estimate_cost(
    prompt: str,
    expected_output_tokens: int,
    input_price_per_1m: float,
    output_price_per_1m: float,
    tokenizer_encode=None,
) -> dict:
    '''Estimate the API cost for a single request.'''
    if tokenizer_encode is None:
        tokenizer_encode = gpt4_enc.encode
    input_tokens = len(tokenizer_encode(prompt))
    input_cost = (input_tokens / 1_000_000) * input_price_per_1m
    output_cost = (expected_output_tokens / 1_000_000) * output_price_per_1m
    return {
        "input_tokens": input_tokens,
        "output_tokens": expected_output_tokens,
        "total_tokens": input_tokens + expected_output_tokens,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": input_cost + output_cost,
    }

# ── Example ───────────────────────────────────────────────────────
sample_prompt = (
    "You are a helpful assistant. Summarize the following article in 3 bullet points.\n\n"
    + "The European Central Bank held interest rates steady on Thursday, " * 20
)

for model_name, inp_price, out_price in [
    ("GPT-4o", 2.50, 10.00),
    ("GPT-4o-mini", 0.15, 0.60),
    ("Qwen3-235B", 0.70, 2.80),
]:
    result = estimate_cost(sample_prompt, 200, inp_price, out_price)
    print(
        f"  {model_name:>15s}:  {result['input_tokens']:>5d} in + "
        f"{result['output_tokens']:>4d} out = {result['total_tokens']:>5d} total  │  "
        f"${result['total_cost']:.5f}"
    )

### Context window budgeting

A 128K-token context window sounds enormous, but it fills quickly once you add a system prompt, few-shot examples, retrieved documents, and leave room for generation. Think of it as a fixed budget:

```
┌──────────────────────────────────────────────────────┐
│                  128,000 tokens                       │
├──────────┬───────────┬──────────────┬────────────────┤
│ System   │ User msg  │ Retrieved    │ Generation     │
│ prompt   │           │ context      │ (reserved)     │
│ ~500     │ ~200      │ ~120,000     │ ~7,300         │
└──────────┴───────────┴──────────────┴────────────────┘
```

The retrieved context block dominates. If your RAG pipeline retrieves too many chunks, you'll starve the generation budget. Let's visualize this trade-off.

In [ ]:
# @title 4.2 — Context budget visualizer

def plot_context_budget(
    context_window: int,
    system_tokens: int,
    user_tokens: int,
    retrieved_tokens: int,
    generation_reserve: int,
    title: str = "Context window budget",
) -> None:
    '''Bar chart showing how the context window is allocated.'''
    labels = ["System\nprompt", "User\nmessage", "Retrieved\ncontext", "Generation\n(reserved)"]
    values = [system_tokens, user_tokens, retrieved_tokens, generation_reserve]
    total_used = sum(values)
    remaining = context_window - total_used
    if remaining > 0:
        labels.append("Unused")
        values.append(remaining)
    elif remaining < 0:
        labels.append("⚠️ OVERFLOW")
        values.append(-remaining)

    colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974" if remaining >= 0 else "#FF0000"]
    colors = colors[: len(values)]

    fig, ax = plt.subplots(figsize=(10, 1.8))
    left = 0
    for lbl, val, col in zip(labels, values, colors):
        ax.barh(0, val, left=left, color=col, edgecolor="white", height=0.6)
        if val > context_window * 0.04:
            ax.text(
                left + val / 2, 0, f"{lbl}\n{val:,}",
                ha="center", va="center", fontsize=8, color="white", fontweight="bold",
            )
        left += val

    ax.set_xlim(0, max(context_window, total_used))
    ax.set_yticks([])
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x / 1000:.0f}K"))
    ax.set_xlabel("Tokens")
    ax.set_title(title, fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

# ── Scenario: RAG with 128K context ──────────────────────────────
plot_context_budget(
    context_window=128_000,
    system_tokens=500,
    user_tokens=200,
    retrieved_tokens=120_000,
    generation_reserve=7_300,
    title="128K context — RAG-heavy allocation",
)

# ── Scenario: Chat with 8K context ───────────────────────────────
plot_context_budget(
    context_window=8_192,
    system_tokens=400,
    user_tokens=1_500,
    retrieved_tokens=0,
    generation_reserve=4_000,
    title="8K context — Simple chat allocation",
)

## 5 — Tokenization gotchas: what can go wrong

Tokenization bugs are insidious because they're invisible — your prompt *looks* fine to a human reader, but the model sees something completely different. Here are the failure modes every AI engineer should know.

### 5.1 — Number tokenization: why LLMs are bad at arithmetic

When you ask an LLM "What is 12345 + 67890?", the model doesn't see the numbers the way you do. The tokenizer might split them like this:

- `"12345"` → `["123", "45"]`
- `"67890"` → `["678", "90"]`

The model must somehow learn that `["123", "45"]` means 12,345 and not "123 followed by 45." This is a known bottleneck for arithmetic, symbolic math, and any task that requires precise numerical reasoning. Some models add special handling for digits (one token per digit), but most general tokenizers don't.

In [ ]:
# @title 5.1 — Number tokenization demo

numbers = ["42", "1234", "12345", "123456789", "3.14159265", "1.23e-7"]

print("How tokenizers split numbers:\n")
for num in numbers:
    q_pieces = [qwen_tok.decode([t]) for t in qwen_tok.encode(num)]
    g_pieces = [gpt4_enc.decode([t]) for t in gpt4_enc.encode(num)]
    l_pieces = [llama_tok.decode([t]) for t in llama_tok.encode(num, add_special_tokens=False)]
    print(f'  "{num}"')
    print(f"    Qwen3 : {q_pieces}")
    print(f"    GPT-4 : {g_pieces}")
    print(f"    Llama : {l_pieces}")
    print()

### 5.2 — The leading-space trap

Most BPE tokenizers treat `" Hello"` (with a leading space) as a different token than `"Hello"`. This is by design — in natural text, words are usually preceded by a space, so `" Hello"` is the common form. But it means:

- If you programmatically build prompts by concatenating strings, you might accidentally create tokens the model rarely saw during training.
- Stripping whitespace from user input before inserting it into a template can change the tokenization in subtle ways.

In [ ]:
# @title 5.2 — Leading space matters

for text in ["Hello", " Hello", "  Hello"]:
    ids = gpt4_enc.encode(text)
    pieces = [gpt4_enc.decode([t]) for t in ids]
    print(f"  {repr(text):>12s}  →  {ids}  →  {pieces}")

### 5.3 — Multilingual inefficiency

The same *meaning* expressed in different languages can require vastly different numbers of tokens. This is a direct consequence of training data distribution: if 60% of BPE training data is English, English subwords will be well-compressed, while Hindi, Thai, or Arabic text will be broken into many small tokens.

In [ ]:
# @title 5.3 — Multilingual token efficiency

# "The weather is nice today" in several languages
translations = {
    "English":    "The weather is nice today",
    "Spanish":    "El clima es agradable hoy",
    "Chinese":    "今天天气很好",
    "Japanese":   "今日はいい天気ですね",
    "Arabic":     "الطقس جميل اليوم",
    "Hindi":      "आज मौसम अच्छा है",
    "Korean":     "오늘 날씨가 좋아요",
    "Thai":       "วันนี้อากาศดี",
}

rows = []
for lang, text in translations.items():
    q = len(qwen_tok.encode(text))
    g = len(gpt4_enc.encode(text))
    l = len(llama_tok.encode(text, add_special_tokens=False))
    rows.append((lang, q, g, l, len(text)))

print(f"{'Language':>10s}  │ {'Qwen3':>5s} │ {'GPT-4':>5s} │ {'Llama':>5s} │ {'Chars':>5s} │ {'Tokens/char (GPT-4)':>18s}")
print("─" * 75)
for lang, q, g, l, c in rows:
    ratio = g / c if c > 0 else 0
    print(f"{lang:>10s}  │ {q:>5d} │ {g:>5d} │ {l:>5d} │ {c:>5d} │ {ratio:>18.2f}")

### 5.4 — Code tokenization: indentation and syntax

Code presents unique challenges for tokenizers:

- **Indentation**: Python's 4-space indent might be one token or four separate space tokens, depending on the tokenizer. This can account for 10–15% of total tokens in Python code.
- **Identifiers**: Variable names like `my_variable_name` get split at different points by different tokenizers.
- **Operators**: Some tokenizers merge common operator sequences (`:=`, `->`, `**`) into single tokens; others don't.

In [ ]:
# @title 5.4 — Code tokenization: Python indentation

python_code = '''def calculate_total(items: list[dict]) -> float:
    \"\"\"Sum the prices of all items.\"\"\"
    total = 0.0
    for item in items:
        if item.get("active", False):
            total += item["price"] * item["quantity"]
    return total
'''

for name, encode_fn in [
    ("Qwen3", lambda t: qwen_tok.encode(t)),
    ("GPT-4", lambda t: gpt4_enc.encode(t)),
    ("Llama", lambda t: llama_tok.encode(t, add_special_tokens=False)),
]:
    ids = encode_fn(python_code)
    print(f"  {name:>6s}: {len(ids)} tokens for {len(python_code)} chars "
          f"(ratio: {len(ids)/len(python_code):.2f} tokens/char)")

# Show how indentation specifically tokenizes
indent = "        "  # 8 spaces (2 levels of Python indent)
print(f"\n  How 8 spaces tokenize:")
for name, enc_fn, dec_fn in [
    ("Qwen3", qwen_tok.encode, lambda t: qwen_tok.decode([t])),
    ("GPT-4", gpt4_enc.encode, lambda t: gpt4_enc.decode([t])),
    ("Llama", lambda t: llama_tok.encode(t, add_special_tokens=False),
             lambda t: llama_tok.decode([t])),
]:
    ids = enc_fn(indent)
    pieces = [repr(dec_fn(t)) for t in ids]
    print(f"    {name:>6s}: 8 spaces → {len(ids)} token(s): {pieces}")

## 6 — Practical rules for AI engineers

Tokenization knowledge turns abstract into actionable. Here are the rules that experienced AI engineers follow:

**Rule 1: Always check tokenization before assuming a prompt fits in context.**
Don't eyeball it. Count tokens programmatically. A 4,000-word essay might be 5,000 or 8,000 tokens depending on the content and tokenizer.

**Rule 2: Budget tokens explicitly.**
Allocate your context window like a financial budget:
```
context_window = system_prompt + user_input + retrieved_chunks + generation_reserve
```
If you don't reserve room for generation, the model's output will be truncated.

**Rule 3: Know your tokenizer's vocabulary size.**
This determines the embedding matrix dimensions and affects model memory.

| Model | Vocab size | Embedding dims |
|-------|-----------|----------------|
| GPT-4 (cl100k) | 100,257 | Not public |
| Llama-3.1-8B | ~128,256 | 4,096 |
| Qwen3-8B | ~151,643 | 4,096 |

**Rule 4: Multilingual? Check token efficiency for your target languages.**
If you're building a product for Japanese users, measure the token overhead vs. English and factor it into cost projections.

**Rule 5: Structured output (JSON) is token-expensive.**
JSON's braces, colons, quotes, and field names consume tokens. A JSON response can be 30–50% longer in tokens than the equivalent plain-text answer.

In [ ]:
# @title 6.1 — JSON vs plain-text token cost comparison

plain_text = (
    "Alice scored 98, 87, and 95. Her average is 93.3. "
    "She is currently an active student."
)

json_text = json.dumps({
    "name": "Alice",
    "scores": [98, 87, 95],
    "average": 93.3,
    "active": True,
}, indent=2)

for label, text in [("Plain text", plain_text), ("JSON", json_text)]:
    tok_count = len(gpt4_enc.encode(text))
    print(f"  {label:>10s}: {tok_count:>4d} tokens  ({len(text):>4d} chars)")

overhead = len(gpt4_enc.encode(json_text)) / len(gpt4_enc.encode(plain_text))
print(f"\n  JSON overhead: {overhead:.1f}× more tokens for equivalent information")

In [ ]:
# @title 6.2 — Vocabulary sizes

print("Tokenizer vocabulary sizes:\n")
print(f"  Qwen3-8B:          {qwen_tok.vocab_size:>10,d} tokens")
print(f"  GPT-4 (cl100k):    {gpt4_enc.n_vocab:>10,d} tokens")
print(f"  Llama-3.1-8B:      {llama_tok.vocab_size:>10,d} tokens")
print()
print("Larger vocabulary → more single-token representations → shorter sequences,")
print("but also a larger embedding matrix (vocab_size × hidden_dim).")

## Exercises

### Exercise 1 — BPE merge experiment

Take the `run_bpe()` function from Section 2 and run it on a different corpus. Try using words from a technical domain (e.g., medical terms, programming keywords, or a foreign language).

**Tasks**:
1. Construct a corpus of at least 20 words (with repetitions to simulate frequency).
2. Run BPE with 15 merges.
3. Compare the resulting vocabulary with the one we built earlier. How do the merges differ?
4. Try encoding a word that was *not* in the training corpus. Does it decompose into meaningful subwords?

In [ ]:
# @title Exercise 1 — Starter code

# YOUR CORPUS HERE — replace with domain-specific words
exercise_corpus = [
    # Example: programming keywords (adjust frequencies by repeating)
    "function", "function", "function", "function",
    "variable", "variable", "variable",
    "constant", "constant",
    "return", "return", "return", "return", "return",
    "parameter", "parameter", "parameter",
    "argument", "argument",
    "exception", "exception", "exception",
    "inheritance", "inheritance",
]

# TODO: Run BPE on this corpus
# final_seqs, merges = run_bpe(exercise_corpus, num_merges=15)

# TODO: Encode an unseen word
# unseen = "functional"
# print(encode_word(unseen, merges))

### Exercise 2 — Token budget calculator

Build a function that takes:
- A system prompt (string)
- A user message (string)
- A list of retrieved text chunks (list of strings)
- A context window size (int)
- A minimum generation reserve (int)

And returns whether the prompt fits, how many tokens are used, and how many chunks can be included.

In [ ]:
# @title Exercise 2 — Starter code

def check_token_budget(
    system_prompt: str,
    user_message: str,
    chunks: list[str],
    context_window: int = 128_000,
    generation_reserve: int = 4_096,
    encode_fn=None,
) -> dict:
    '''Check whether the assembled prompt fits in the context window.

    Returns dict with token counts and how many chunks fit.
    '''
    if encode_fn is None:
        encode_fn = gpt4_enc.encode

    # TODO: Implement this function
    # 1. Count tokens for system_prompt and user_message
    # 2. Iterate through chunks, adding them until you'd exceed the budget
    # 3. Return results
    raise NotImplementedError("Complete this exercise!")

# Test with sample data
# system = "You are a helpful research assistant."
# user = "Summarize the key findings about climate change."
# chunks = ["chunk " * 500] * 10  # 10 large chunks
# result = check_token_budget(system, user, chunks, context_window=8192)
# print(result)

### Exercise 3 — Multilingual token audit

Pick a paragraph of text (e.g., from Wikipedia) and translate it into 5 languages using an online translator. Tokenize each version with all three tokenizers and create a bar chart comparing token counts.

**Tasks**:
1. Prepare the same paragraph in at least 5 languages.
2. Tokenize with Qwen3, GPT-4, and Llama.
3. Plot a grouped bar chart (languages on x-axis, tokenizer as groups).
4. Write a 3-sentence analysis: which language is most expensive? Which tokenizer is most efficient across languages? What does this mean for a multilingual product?

In [ ]:
# @title Exercise 3 — Starter code

# TODO: Replace with your translations
paragraphs = {
    "English": "Climate change is one of the most pressing challenges of our time.",
    "Spanish": "El cambio climático es uno de los desafíos más urgentes de nuestro tiempo.",
    "Chinese": "气候变化是我们这个时代最紧迫的挑战之一。",
    "Arabic":  "تغير المناخ هو أحد أكثر التحديات إلحاحًا في عصرنا.",
    "Hindi":   "जलवायु परिवर्तन हमारे समय की सबसे गंभीर चुनौतियों में से एक है।",
}

# TODO: Tokenize each with all 3 tokenizers
# TODO: Plot a grouped bar chart using matplotlib
# Hint: use plt.bar() with offset x positions for each tokenizer group

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Tokenization is the invisible first step that determines cost, context fit, and model capability. |
| 2 | BPE works by iteratively merging the most frequent byte pairs, building a vocabulary of subwords that balances coverage and sequence length. |
| 3 | Different tokenizers (tiktoken, SentencePiece, Qwen) produce different token counts for the same text — always measure, don't assume. |
| 4 | Multilingual and code text can be 2–5× more expensive in tokens than English prose. |
| 5 | Always budget your context window explicitly: system prompt + user input + retrieval + generation reserve. |
| 6 | Numbers, leading whitespace, emojis, and JSON are common sources of tokenization surprises. |

## What's Next

- **Next**: [02_embeddings.ipynb](02_embeddings.ipynb) — How tokens become dense vectors that encode meaning, and why embedding quality is the foundation of RAG, search, and classification.
- **Related**: [03_attention_and_transformers.ipynb](03_attention_and_transformers.ipynb) — How the transformer architecture processes token sequences through self-attention.

## References & Further Reading

1. [Sennrich, Haddow, & Birch, "Neural Machine Translation of Rare Words with Subword Units," 2016](https://arxiv.org/abs/1508.07909) — The original BPE-for-NLP paper. Essential reading for understanding why subword tokenization works.
2. [Kudo & Richardson, "SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing," 2018](https://arxiv.org/abs/1808.06226) — Introduces the SentencePiece framework and the Unigram language model approach to tokenization.
3. [OpenAI tiktoken documentation](https://github.com/openai/tiktoken) — Reference implementation and encoding definitions for GPT-family tokenizers.
4. [Qwen3 Technical Report](https://arxiv.org/abs/2505.09388) — Details on the Qwen3 tokenizer design, including byte-fallback BPE and the expanded vocabulary for CJK and code.
5. [Hugging Face Tokenizers documentation](https://huggingface.co/docs/tokenizers/) — Comprehensive guide to the `tokenizers` library used under the hood by most Hugging Face models.
6. [Petrov et al., "Language Model Tokenizers Introduce Unfairness Between Languages," 2023](https://arxiv.org/abs/2305.15425) — Quantifies the multilingual token inefficiency problem and its downstream effects on cost and capability.